
# Bronze Layer Ingestion Notebook
PE Fund Capital, Payment & Reconciliation Intelligence Platform

Handles all 10 sources:
- 6 internal CSV sources (straightforward schema enforcement)
- 4 external `.dat` sources (pipe-delimited, HDR/TRL wrapped, quarantine on malformed rows)

Called by ADF with 3 widget parameters: `source_name`, `folder_path`, `run_id`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, LongType, DateType, TimestampType
)
from datetime import datetime

## Widgets

In [ ]:
dbutils.widgets.text("source_name", "")
dbutils.widgets.text("folder_path", "")
dbutils.widgets.text("run_id", "")


source_name = dbutils.widgets.get("source_name").strip()
folder_path = dbutils.widgets.get("folder_path").strip()
run_id = dbutils.widgets.get("run_id").strip()

if not source_name or not folder_path or not run_id:
    raise ValueError(
        f"Missing required widget value(s): source_name='{source_name}', "
        f"folder_path='{folder_path}', run_id='{run_id}'"
    )

print(f"source_name = {source_name}")
print(f"folder_path = {folder_path}")
print(f"run_id      = {run_id}")

## Storage config — Unity Catalog External Location

`/mnt/bronze/...` (DBFS mount) is NOT used here — serverless compute
does not support legacy DBFS mounts or `fs.azure.account.key` config.
Bronze is written via a direct `abfss://` path, backed by a Unity
Catalog External Location (Storage Credential -> Access Connector ->
Managed Identity). No keys involved.


In [ ]:
STORAGE_ACCOUNT = "pestorage3"
BRONZE_CONTAINER = "bronze"

BRONZE_BASE = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print(f"Bronze base path = {BRONZE_BASE}")


## Config — Part 1: internal CSV sources

In [ ]:
# Each entry: column name -> Spark SQL type string used for casting.
INTERNAL_SCHEMAS = {
    "investor": {
        "investor_id": "string",
        "investor_name": "string",
        "investor_type": "string",
        "country": "string",
    },
    "fund": {
        "fund_id": "string",
        "fund_name": "string",
        "vintage_year": "int",
        "fund_size_usd": "double",
    },
    "commitment": {
        "commitment_id": "string",
        "fund_id": "string",
        "investor_id": "string",
        "committed_amount_usd": "double",
        "commitment_date": "date",
        "contributed_total": "double",
        "invested_total": "double",
    },
    "portfolio_company": {
        "company_id": "string",
        "company_name": "string",
        "fund_id": "string",
        "industry": "string",
        "benchmark_ticker": "string",  # nullable
    },
    "payment": {
        "payment_id": "string",
        "commitment_id": "string",
        "fund_id": "string",
        "investor_id": "string",
        "company_id": "string",  # nullable
        "payment_type": "string",
        "amount_usd": "double",
        "status": "string",
        "event_date": "date",
        "entry_benchmark_price": "double",  # nullable
    },
    "market_price": {
        "ticker": "string",
        "bar_timestamp": "timestamp",
        "open": "double",
        "high": "double",
        "low": "double",
        "close": "double",
        "volume": "long",
    },
}

## Config — Part 2: external `.dat` sources

In [ ]:
# record_type: the tag at the start of each data line inside the .dat file
# columns: ordered list of (name, type) matching the pipe-delimited layout,
#          in the order they appear AFTER the record-type tag
EXTERNAL_SCHEMAS = {
    "external_position": {
        "record_type": "POS",
        "columns": [
            ("position_id", "string"),
            ("fund_id", "string"),
            ("asset_id", "string"),
            ("quantity", "double"),
            ("price", "double"),
            ("currency", "string"),
            ("timestamp", "timestamp"),
            ("source_system", "string"),
            ("status", "string"),
        ],
    },
    "external_cash": {
        "record_type": "CASH",
        "columns": [
            ("cash_id", "string"),
            ("fund_id", "string"),
            ("currency", "string"),
            ("amount", "double"),
            ("timestamp", "timestamp"),
            ("source_system", "string"),
            ("status", "string"),
        ],
    },
    "external_payment": {
        "record_type": "PMT",
        "columns": [
            ("payment_id", "string"),
            ("fund_id", "string"),
            ("payment_type", "string"),
            ("amount", "double"),
            ("currency", "string"),
            ("status", "string"),
            ("event_timestamp", "timestamp"),
            ("settlement_timestamp", "timestamp"),
            ("source_system", "string"),
        ],
    },
    "external_reference": {
        "record_type": "REF",
        "columns": [
            ("ref_id", "string"),
            ("asset_id", "string"),
            ("asset_name", "string"),
            ("identifier", "string"),
            ("asset_type", "string"),
            ("currency", "string"),
            ("status", "string"),
            ("effective_date", "string"),  # raw format e.g. 20260915, kept as string in Bronze
            ("timestamp", "timestamp"),
            ("source_system", "string"),
        ],
    },
}

# Fields that must be present, numeric, and >= 0 (negative triggers NEGATIVE_<FIELD>,
# empty/null triggers NULL_<FIELD>, non-numeric triggers INVALID_<FIELD>).
# Only fields explicitly validated per the spec's known test cases; extend as needed.
NUMERIC_NONNEGATIVE_FIELDS = {
    "external_position": ["quantity"],
}
NUMERIC_FIELDS = {
    "external_position": ["quantity", "price"],
    "external_cash": ["amount"],
    "external_payment": ["amount"],
}
TIMESTAMP_FIELDS = {
    "external_position": ["timestamp"],
    "external_cash": ["timestamp"],
    "external_payment": ["event_timestamp", "settlement_timestamp"],
    "external_reference": ["timestamp"],
}

## Part 1 — internal CSV ingestion

In [ ]:
def ingest_internal(source_name, folder_path, run_id):
    schema = INTERNAL_SCHEMAS[source_name]

    df = spark.read.option("header", True).csv(folder_path)

    missing = set(schema.keys()) - set(df.columns)
    if missing:
        raise ValueError(f"[{source_name}] CSV is missing expected columns: {sorted(missing)}")

    for col_name, col_type in schema.items():
        df = df.withColumn(col_name, F.col(col_name).cast(col_type))

    df = df.select(*schema.keys())
    df = df.withColumn("_run_id", F.lit(run_id)) \
           .withColumn("_ingested_at", F.current_timestamp())

    out_path = f"{BRONZE_BASE}/{source_name}/"
    df.write.format("delta").mode("overwrite").save(out_path)

    print(f"[{source_name}] wrote {df.count()} rows to {out_path}")
    return df

## Part 2 — external `.dat` ingestion (HDR/TRL, pipe-delimited, quarantine)

In [ ]:
def _cast_value(raw, col_type):
    """Return (casted_value, error_reason_or_None) for a single field."""
    raw = raw.strip() if raw is not None else raw

    if col_type == "double":
        if raw is None or raw == "":
            return None, "NULL"
        try:
            val = float(raw)
        except ValueError:
            return None, "INVALID"
        return val, None

    if col_type == "timestamp":
        if raw is None or raw == "":
            return None, "NULL"
        try:
            # Expected format: 2026-09-15T09:15:00
            parsed = datetime.strptime(raw, "%Y-%m-%dT%H:%M:%S")
        except ValueError:
            return None, "INVALID"
        return parsed, None

    # string / other passthrough types
    return raw, None


def _validate_row(source_name, field_dict):
    """
    field_dict: {column_name: raw_string_value}
    Returns list of reason codes (empty list = row is valid).
    """
    reasons = []
    numeric_fields = NUMERIC_FIELDS.get(source_name, [])
    nonneg_fields = set(NUMERIC_NONNEGATIVE_FIELDS.get(source_name, []))
    ts_fields = TIMESTAMP_FIELDS.get(source_name, [])

    for field in numeric_fields:
        raw = field_dict.get(field)
        val, err = _cast_value(raw, "double")
        if err == "NULL":
            reasons.append(f"NULL_{field.upper()}")
        elif err == "INVALID":
            reasons.append(f"INVALID_{field.upper()}")
        elif field in nonneg_fields and val is not None and val < 0:
            reasons.append(f"NEGATIVE_{field.upper()}")

    for field in ts_fields:
        raw = field_dict.get(field)
        _, err = _cast_value(raw, "timestamp")
        if err == "NULL":
            reasons.append(f"NULL_{field.upper()}")
        elif err == "INVALID":
            reasons.append(f"INVALID_{field.upper()}")

    return reasons


def ingest_external(source_name, folder_path, run_id):
    config = EXTERNAL_SCHEMAS[source_name]
    record_type = config["record_type"]
    columns = config["columns"]  # list of (name, type)
    col_names = [c for c, _ in columns]

    raw_lines = spark.read.text(folder_path).collect()
    lines = [r.value for r in raw_lines if r.value is not None and r.value != ""]

    if not lines:
        raise ValueError(f"[{source_name}] no lines found at {folder_path}")

    hdr_line = None
    trl_line = None
    data_lines = []

    for line in lines:
        parts = line.split("|")
        tag = parts[0]
        if tag == "HDR":
            hdr_line = parts
        elif tag == "TRL":
            trl_line = parts
        elif tag == record_type:
            data_lines.append(parts)
        # any other tag is ignored/unexpected — could be logged if needed

    if hdr_line is None:
        raise ValueError(f"[{source_name}] file has no HDR line: {folder_path}")
    if trl_line is None:
        raise ValueError(f"[{source_name}] file has no TRL line: {folder_path}")

    # TRL format: TRL|<record_count>|TOTAL_RECORDS
    stated_count = int(trl_line[1])
    actual_count = len(data_lines)
    count_mismatch = stated_count != actual_count
    if count_mismatch:
        print(
            f"[{source_name}] *** RECORD COUNT MISMATCH *** "
            f"TRL stated={stated_count}, actual parsed={actual_count}. "
            f"Flagging file for review."
        )

    valid_rows = []
    quarantine_rows = []

    for parts in data_lines:
        values = parts[1:]  # drop the record-type tag
        original_line = "|".join(parts)

        if len(values) != len(col_names):
            quarantine_rows.append({
                "raw_row": original_line,
                "reason_code": "FIELD_COUNT_MISMATCH",
                "run_id": run_id,
            })
            continue

        field_dict = dict(zip(col_names, values))
        reasons = _validate_row(source_name, field_dict)

        if reasons:
            quarantine_rows.append({
                "raw_row": original_line,
                "reason_code": ",".join(reasons),
                "run_id": run_id,
            })
            continue

        # Row is clean — cast every field to its target type for Bronze output
        casted = {}
        for col_name, col_type in columns:
            val, _ = _cast_value(field_dict[col_name], col_type)
            casted[col_name] = val
        casted["_run_id"] = run_id
        casted["_source_file_record_count_mismatch"] = count_mismatch
        valid_rows.append(casted)

    # Build Delta output for valid rows
    if valid_rows:
        valid_schema = StructType(
            [StructField(c, _spark_type(t), True) for c, t in columns]
            + [
                StructField("_run_id", StringType(), True),
                StructField("_source_file_record_count_mismatch", StringType(), True),
            ]
        )
        valid_df = spark.createDataFrame(valid_rows, schema=valid_schema)
    else:
        valid_schema = StructType(
            [StructField(c, _spark_type(t), True) for c, t in columns]
            + [
                StructField("_run_id", StringType(), True),
                StructField("_source_file_record_count_mismatch", StringType(), True),
            ]
        )
        valid_df = spark.createDataFrame([], schema=valid_schema)

    valid_df = valid_df.withColumn("_ingested_at", F.current_timestamp())
    out_path = f"{BRONZE_BASE}/{source_name}/"
    valid_df.write.format("delta").mode("overwrite").save(out_path)
    print(f"[{source_name}] wrote {valid_df.count()} valid rows to {out_path}")

    # Build Delta output for quarantined rows
    q_schema = StructType([
        StructField("raw_row", StringType(), True),
        StructField("reason_code", StringType(), True),
        StructField("run_id", StringType(), True),
    ])
    if quarantine_rows:
        q_df = spark.createDataFrame(quarantine_rows, schema=q_schema)
    else:
        q_df = spark.createDataFrame([], schema=q_schema)

    q_df = q_df.withColumn("_ingested_at", F.current_timestamp())
    q_path = f"{BRONZE_BASE}/_quarantine/{source_name}/"
    q_df.write.format("delta").mode("overwrite").save(q_path)
    print(f"[{source_name}] wrote {q_df.count()} quarantined rows to {q_path}")

    return valid_df, q_df


def _spark_type(type_str):
    return {
        "string": StringType(),
        "double": DoubleType(),
        "int": IntegerType(),
        "long": LongType(),
        "date": DateType(),
        "timestamp": TimestampType(),
    }[type_str]

## Dispatch — run the right path based on `source_name`

In [ ]:
if source_name in INTERNAL_SCHEMAS:
    ingest_internal(source_name, folder_path, run_id)
elif source_name in EXTERNAL_SCHEMAS:
    ingest_external(source_name, folder_path, run_id)
else:
    raise ValueError(
        f"Unknown source_name '{source_name}'. Expected one of: "
        f"{sorted(list(INTERNAL_SCHEMAS.keys()) + list(EXTERNAL_SCHEMAS.keys()))}"
    )

## Unit tests — using the pack's known-bad rows (`POSITION_..._INVALID.dat`)

These exercise `_validate_row` directly against the three documented test
cases, without needing Spark or a real file. Run this cell standalone to
verify the quarantine logic before wiring it into a job.

In [ ]:
def _run_known_bad_row_tests():
    cols = [c for c, _ in EXTERNAL_SCHEMAS["external_position"]["columns"]]

    test_cases = [
        {
            "name": "BAD001 - empty quantity",
            "line": "POS|BAD001|FND001|AST001||125.50|USD|2026-09-17T11:00:00|CUSTODIAN_A|VALID",
            "expect_quarantined": True,
            "expect_reasons_contains": ["NULL_QUANTITY"],
        },
        {
            "name": "BAD002 - negative qty, bad price, bad timestamp",
            "line": "POS|BAD002|FNDXXX|AST999|-500|ABC|USD|INVALID_TIMESTAMP|CUSTODIAN_A|UNKNOWN",
            "expect_quarantined": True,
            "expect_reasons_contains": ["NEGATIVE_QUANTITY", "INVALID_PRICE", "INVALID_TIMESTAMP"],
        },
        {
            "name": "BAD003 - actually valid row",
            "line": "POS|BAD003|FND002|AST004|7000|103.00|USD|2026-09-17T11:05:00|CUSTODIAN_A|VALID",
            "expect_quarantined": False,
            "expect_reasons_contains": [],
        },
    ]

    all_passed = True
    for case in test_cases:
        parts = case["line"].split("|")
        values = parts[1:]
        field_dict = dict(zip(cols, values))
        reasons = _validate_row("external_position", field_dict)

        is_quarantined = len(reasons) > 0
        reasons_ok = all(r in reasons for r in case["expect_reasons_contains"])
        passed = (is_quarantined == case["expect_quarantined"]) and reasons_ok

        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {case['name']} -> reasons={reasons}")
        all_passed = all_passed and passed

    print("\nALL TESTS PASSED" if all_passed else "\nSOME TESTS FAILED")
    return all_passed


